# Dataset rescue statuses

Run `Resolver.get_rescue` concurrently for the machine-readable endpoints listed in the wildfire data-source notes. The URL comments point back to the corresponding dataset webpage.


In [1]:
import asyncio
import sys
from pathlib import Path

import httpx
import pandas as pd

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "ptps_wildfire_demo").is_dir()
)
sys.path.insert(0, str(repo_root))

from ptps_wildfire_demo.proxy.resolver import Resolver

In [ ]:
datasets = [
    {
        "name": "Source Cooperative USGS MTBS burned perimeters",
        "webpage": "https://source.coop/cboettig/fire",
        "url": "https://data.source.coop/cboettig/fire/usgs-mtbs.parquet",
    },
    {
        "name": "Source Cooperative weather observations",
        "webpage": "https://source.coop/bkr/obs",
        "url": "https://data.source.coop/bkr/obs/sigmets_200501010000_202604010000.parquet",
    },
    {
        "name": "Source Cooperative National Wetlands Inventory",
        "webpage": "https://source.coop/giswqs/nwi",
        "url": "https://data.source.coop/giswqs/nwi/wetlands/DC_Wetlands.parquet",
    },
    {
        "name": "USFS probabilistic wildfire risk burn probability",
        "webpage": "https://data-usfs.hub.arcgis.com/datasets/usfs::probabilistic-wildfire-risk-burn-probability-image-service/explore",
        "url": "https://apps.fs.usda.gov/fsgisx01/rest/services/Enterprise/Probabilistic_Wildfire_Risk/MapServer",
    },
    {
        "name": "Wildfire Risk to Communities",
        "webpage": "https://wildfirerisk.org/download/",
        "url": "https://wildfirerisk.org/wp-content/uploads/2026/04/wrc_download_20260415.xlsx",
    },
    {
        "name": "CarbonPlan Open Climate Risk buildings",
        "webpage": "https://source.coop/carbonplan/carbonplan-ocr",
        "url": "https://s3.us-west-2.amazonaws.com/us-west-2.opendata.source.coop/carbonplan/carbonplan-ocr/output/fire-risk/vector/production/v1.1.0/geoparquet/buildings.parquet",
    },
    {
        "name": "NOAA weather alerts",
        "webpage": "https://www.weather.gov/documentation/services-web-api",
        "url": "https://api.weather.gov/alerts/active?event=Red%20Flag%20Warning",
    },
    {
        "name": "NOAA HMS fire and smoke product",
        "webpage": "https://www.ospo.noaa.gov/products/land/hms.html",
        "url": "https://www.ospo.noaa.gov/data/hms/fire/",
    },
    {
        "name": "NOAA Storm Events Database",
        "webpage": "https://www.ncdc.noaa.gov/stormevents/",
        "url": "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260819.csv.gz",
    },
    {
        "name": "NASA FIRMS active fire detections",
        "webpage": "https://firms.modaps.eosdis.nasa.gov/",
        "url": "https://firms.modaps.eosdis.nasa.gov/active_fire/",
    },
    {
        "name": "WFIGS current interagency fire perimeters",
        "webpage": "https://data-nifc.opendata.arcgis.com/maps/d1c32af3212341869b3c810f1a215824",
        "url": "https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Interagency_Perimeters_Current/FeatureServer",
    },
    {
        "name": "MODIS NDVI granules",
        "webpage": "https://modis.gsfc.nasa.gov/data/dataprod/mod13.php",
        "url": "https://cmr.earthdata.nasa.gov/search/granules.json?short_name=MOD13Q1&page_size=1",
    },
    {
        "name": "OpenStreetMap infrastructure",
        "webpage": "https://overpass-api.de/",
        "url": "https://overpass-api.de/api/interpreter?data=%5Bout%3Ajson%5D%3Bway%5Bhighway%5D%2845.4%2C-122.8%2C45.5%2C-122.6%29%3Bout%20geom%3B",
    },
    {
        "name": "USDA LANDFIRE seasonal fuels",
        "webpage": "https://www.landfire.gov/fuel/seasonal_fuels",
        "url": "https://www.landfire.gov/sites/default/files/CSV/LF2025/LF2025_FBFM40.csv",
    },
]

pd.DataFrame(datasets)

,name,url
0,Source Cooperative USGS MTBS burned perimeters,https://data.source.coop/cboettig/fire/usgs-mtbs.parquet
1,Source Cooperative weather observations,https://data.source.coop/bkr/obs/sigmets_200501010000_202604010000.parquet
2,Source Cooperative National Wetlands Inventory,https://data.source.coop/giswqs/nwi/wetlands/DC_Wetlands.parquet
3,USFS probabilistic wildfire risk burn probability,https://apps.fs.usda.gov/fsgisx01/rest/services/Enterprise/Probabilistic_Wildfire_Risk/MapServer
4,Wildfire Risk to Communities,https://wildfirerisk.org/wp-content/uploads/2026/04/wrc_download_20260415.xlsx
5,CarbonPlan Open Climate Risk buildings,https://s3.us-west-2.amazonaws.com/us-west-2.opendata.source.coop/carbonplan/carbonplan-ocr/output/fire-risk/vector/production/v1.1.0/geoparquet/buildings.parquet
6,NOAA weather alerts,https://api.weather.gov/alerts/active?event=Red%20Flag%20Warning
7,NOAA HMS fire and smoke product,https://www.ospo.noaa.gov/data/hms/fire/
8,NOAA Storm Events Database,https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260819.csv.gz
9,NASA FIRMS active fire detections,https://firms.modaps.eosdis.nasa.gov/active_fire/


In [ ]:
async def get_rescues() -> list[dict[str, str | None]]:
    timeout = httpx.Timeout(30.0, connect=30.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        resolver = Resolver(client)
        rescues = await asyncio.gather(
            *(resolver.get_rescue(dataset["url"]) for dataset in datasets)
        )

    return [
        {
            "name": dataset["name"],
            "url": dataset["url"],
            "wayback_newest_url": rescue.wayback_newest_url,
            "drp_metadata_url": rescue.drp_metadata_url,
            "drp_download_location": rescue.drp_download_location,
        }
        for dataset, rescue in zip(datasets, rescues, strict=True)
    ]


results = pd.DataFrame(await get_rescues())
pd.set_option("display.max_colwidth", 200)
results

,name,url,wayback_newest_url,drp_metadata_url,drp_download_location
0,Source Cooperative USGS MTBS burned perimeters,https://data.source.coop/cboettig/fire/usgs-mtbs.parquet,NaN,None,None
1,Source Cooperative weather observations,https://data.source.coop/bkr/obs/sigmets_200501010000_202604010000.parquet,NaN,None,None
2,Source Cooperative National Wetlands Inventory,https://data.source.coop/giswqs/nwi/wetlands/DC_Wetlands.parquet,NaN,None,None
3,USFS probabilistic wildfire risk burn probability,https://apps.fs.usda.gov/fsgisx01/rest/services/Enterprise/Probabilistic_Wildfire_Risk/MapServer,NaN,None,None
4,Wildfire Risk to Communities,https://wildfirerisk.org/wp-content/uploads/2026/04/wrc_download_20260415.xlsx,NaN,None,None
5,CarbonPlan Open Climate Risk buildings,https://s3.us-west-2.amazonaws.com/us-west-2.opendata.source.coop/carbonplan/carbonplan-ocr/output/fire-risk/vector/production/v1.1.0/geoparquet/buildings.parquet,NaN,None,None
6,NOAA weather alerts,https://api.weather.gov/alerts/active?event=Red%20Flag%20Warning,NaN,None,None
7,NOAA HMS fire and smoke product,https://www.ospo.noaa.gov/data/hms/fire/,NaN,None,None
8,NOAA Storm Events Database,https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260819.csv.gz,NaN,None,None
9,NASA FIRMS active fire detections,https://firms.modaps.eosdis.nasa.gov/active_fire/,http://web.archive.org/web/20260826051857/https://firms.modaps.eosdis.nasa.gov/active_fire/,None,None
